In [7]:
import pandas as pd
import ast
from collections import Counter

In [8]:
df = pd.read_json("../data/companies.jsonl", lines=True)

In [9]:
df.head()

,website,operational_name,year_founded,address,employee_count,revenue,primary_naics,description,business_model,target_markets,core_offerings,is_public,secondary_naics
0,rompetrol.ro,Rompetrol,1979.0,"{'country_code': 'ro', 'latitude': 44.4775537,...",NaN,1.349824e+10,"{'code': '324110', 'label': 'Petroleum Refiner...",Rompetrol is a Romanian company specialized in...,"[Wholesale, Manufacturing, Business-to-Busines...","[Energy, Industrial, Transportation]","[Fuel Product Distribution, Quality and Safety...",False,None
1,unilever.ro,Unilever,NaN,"{'country_code': 'ro', 'latitude': 44.4761769,...",NaN,1.280207e+10,"{'code': '424690', 'label': 'Other Chemical an...","Unilever South Central Europe SA, DBA Unilever...","[Retail, Business-to-Business, Business-to-Con...",[Consumer Goods],"[Consumer Goods Distribution and Retail, Ingre...",False,None
2,rompetrol.com,Rompetrol,1974.0,"{'country_code': 'ro', 'latitude': 44.47775356...",7000.0,1.266496e+10,"{'code': '324110', 'label': 'Petroleum Refiner...",Rompetrol is a Romanian energy company engaged...,"[Wholesale, Manufacturing, Business-to-Busines...","[Energy, Petrochemicals]","[Sustainable Energy Solutions, Fuel and Energy...",False,None
3,cbre.ro,CBRE Romania,2008.0,"{'country_code': 'ro', 'latitude': 44.4540414,...",92.0,9.568739e+09,"{'code': '531210', 'label': 'Offices of Real E...",CBRE Romania is a Romanian company engaged in ...,"[Service Provider, Business-to-Business, Enter...","[Hospitality, Retail, Industrial, Real Estate,...","[Data Center Real Estate Services, Real Estate...",False,None
4,timacagro.com,TIMAC AGRO,NaN,"{'country_code': 'ro', 'latitude': 44.4792186,...",65.0,8.907855e+09,"{'code': '325311', 'label': 'Nitrogenous Ferti...",TIMAC AGRO S.R.L. is a French agricultural com...,"[Business-to-Business, Enterprise, Manufacturi...",[Agriculture],"[Agricultural Product Retail, Fertilizer Manuf...",False,None


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 477 entries, 0 to 476
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   website           439 non-null    str    
 1   operational_name  475 non-null    str    
 2   year_founded      346 non-null    float64
 3   address           477 non-null    object 
 4   employee_count    289 non-null    float64
 5   revenue           384 non-null    float64
 6   primary_naics     477 non-null    object 
 7   description       477 non-null    str    
 8   business_model    477 non-null    object 
 9   target_markets    477 non-null    object 
 10  core_offerings    477 non-null    object 
 11  is_public         477 non-null    bool   
 12  secondary_naics   11 non-null     object 
dtypes: bool(1), float64(3), object(6), str(3)
memory usage: 45.3+ KB


In [11]:
df.isnull().sum()

website              38
operational_name      2
year_founded        131
address               0
employee_count      188
revenue              93
primary_naics         0
description           0
business_model        0
target_markets        0
core_offerings        0
is_public             0
secondary_naics     466
dtype: int64

In [12]:
df['description'][0]

'Rompetrol is a Romanian company specialized in petroleum refining, petrochemical operations, and the distribution of fuel products. The company operates as a subsidiary of KMG International and manages integrated refineries in Romania, Moldova, Bulgaria, and Georgia, serving the automotive, industrial, and energy sectors. Rompetrol also provides industrial products, wholesale fuel supply, and e-Mobility services, and is certified for quality, health, safety, and environmental management.'

In [13]:
df['address'][0]

"{'country_code': 'ro', 'latitude': 44.4775537, 'longitude': 26.0713416, 'region_name': 'Bucharest', 'town': 'Bucharest'}"

In [14]:
df['primary_naics'][0]

"{'code': '324110', 'label': 'Petroleum Refineries'}"

In [15]:
df[df["operational_name"].isnull()]

,website,operational_name,year_founded,address,employee_count,revenue,primary_naics,description,business_model,target_markets,core_offerings,is_public,secondary_naics
138,icontributetoml.herokuapp.com,NaN,NaN,"{'country_code': 'ca', 'latitude': 45.5031824,...",NaN,162104.0,"{'code': '541715', 'label': 'Research and Deve...",The entity associated with the domain icontrib...,"[Service Provider, Business-to-Business]","[Cybersecurity, Marketing, Biotechnology, Heal...",[ML Algorithm Taxonomy and Selection Criteria ...,False,None
263,seycosmetics.co.uk,NaN,NaN,"{'country_code': 'us', 'latitude': 27.9438291,...",NaN,62599067.0,"{'code': '325620', 'label': 'Toilet Preparatio...","Bodis, LLC is a United States company speciali...","[Business-to-Business, Manufacturing, Wholesale]",[Cosmetics],"[Beauty Cosmetics Wholesale, Beauty Cosmetics ...",False,None


In [16]:
def safe_parse(val):
    if pd.isna(val):
        return None
    if isinstance(val, (dict, list)):
        return val
    try:
        return ast.literal_eval(str(val))
    except (ValueError, SyntaxError):
        return None

In [17]:
df["address_parsed"] = df["address"]. apply(safe_parse)

In [18]:
df["country_code"] = df["address_parsed"].apply(lambda x: x.get("country_code", "unknown") if isinstance(x,dict) else "unknown")

In [19]:
print("Countries in dataset:", df["country_code"].value_counts())

Countries in dataset: country_code
us    86
ch    43
fr    40
cn    35
se    31
gb    27
no    27
ro    26
dk    26
es    19
fi    13
de    12
in    11
nl     8
au     8
ca     8
it     5
ie     4
sg     4
is     4
kr     4
be     3
pl     3
lu     3
at     2
pt     2
br     2
nz     2
jp     2
id     2
gr     2
ru     2
hr     1
lt     1
ua     1
kw     1
vn     1
tr     1
ar     1
eg     1
cl     1
tw     1
hk     1
Name: count, dtype: int64


In [20]:
print(f"Total unique countries: {df['country_code'].nunique()}")

Total unique countries: 43


In [21]:
df["naics_parsed"] = df["primary_naics"].apply(safe_parse)

In [22]:
df["industry_label"] = df["naics_parsed"].apply(lambda x: x.get("label","unknown") if isinstance(x,dict) else "unknown")

In [23]:
print(f"Unique industries: {df['industry_label'].nunique()}")

Unique industries: 101


In [24]:
print(f"Top 25 industries: {df["industry_label"].value_counts().head(25)}")

Top 25 industries: industry_label
Software Publishers                                                                                                   67
Turbine and Turbine Generator Set Units Manufacturing                                                                 44
Computer Systems Design Services                                                                                      30
Pharmaceutical Preparation Manufacturing                                                                              25
Battery Manufacturing                                                                                                 24
Engineering Services                                                                                                  22
Other Basic Inorganic Chemical Manufacturing                                                                          21
Financial Transactions Processing, Reserve, and Clearinghouse Activities                                              1

In [25]:
logistics = df[df["industry_label"].str.contains("freight|logistics|transport|warehouse", case=False)]

In [26]:
print(f"Logistic-related companies: {len(logistics)}")

Logistic-related companies: 2


In [27]:
print(logistics["industry_label"].value_counts())

industry_label
Support Activities for Rail Transportation    1
Warehouse Clubs and Supercenters              1
Name: count, dtype: int64


In [28]:
logistics_desc = df[df["description"].str.contains("logistics|freight|shipping|supply chain|forwarding|transportation|warehousing", case=False)]

In [29]:
print(f"Companies mentioning logistics in description: {len(logistics_desc)}")

Companies mentioning logistics in description: 60


In [30]:
 print(logistics_desc[["operational_name", "country_code", "industry_label"]].to_string())

                   operational_name country_code                                                                              industry_label
2                         Rompetrol           ro                                                                        Petroleum Refineries
3                      CBRE Romania           ro                                                   Offices of Real Estate Agents and Brokers
9                    Fildas Trading           ro                                          Drugs and Druggists' Sundries Merchant Wholesalers
12                           Romgaz           ro                                                                      Natural Gas Extraction
13                    METRO România           ro                                                   General Line Grocery Merchant Wholesalers
15                     Poșta Română           ro                                                                              Postal Service
16           

In [31]:
logistics_ro = logistics_desc[logistics_desc["country_code"] == "ro"]

In [32]:
print(f"Logistics-related in Romania: {len(logistics_ro)}")

Logistics-related in Romania: 11


In [33]:
print(logistics_ro[["operational_name", "industry_label"]].to_string())

               operational_name                                                                              industry_label
2                     Rompetrol                                                                        Petroleum Refineries
3                  CBRE Romania                                                   Offices of Real Estate Agents and Brokers
9                Fildas Trading                                          Drugs and Druggists' Sundries Merchant Wholesalers
12                       Romgaz                                                                      Natural Gas Extraction
13                METRO România                                                   General Line Grocery Merchant Wholesalers
15                 Poșta Română                                                                              Postal Service
16                        OSCAR  Petroleum and Petroleum Products Merchant Wholesalers (except Bulk Stations and Terminals)
18      

In [34]:
df[["employee_count", "revenue","year_founded"]].describe()

,employee_count,revenue,year_founded
count,2.890000e+02,3.840000e+02,346.000000
mean,2.509067e+04,1.146714e+10,1998.303468
std,1.379895e+05,1.282863e+11,33.156595
min,1.000000e+00,7.524000e+03,1856.000000
25%,5.000000e+00,2.483684e+06,1989.000000
50%,3.000000e+01,2.018448e+07,2011.000000
75%,2.603000e+03,1.592155e+09,2021.000000
max,2.100000e+06,2.500389e+12,2025.000000


In [35]:
df["desc_length"] = df["description"].str.len()

In [36]:
df["desc_length"].describe()

count    477.000000
mean     556.075472
std      140.501273
min      121.000000
25%      469.000000
50%      558.000000
75%      648.000000
max      913.000000
Name: desc_length, dtype: float64

In [37]:
offering_counts = df["core_offerings"].apply(lambda x: len(x) if isinstance(x,list) else 0)

In [38]:
print(f"Items per company in core_offerings: {offering_counts.describe()}")

Items per company in core_offerings: count    477.000000
mean      12.266247
std        3.186495
min        1.000000
25%       11.000000
50%       13.000000
75%       15.000000
max       18.000000
Name: core_offerings, dtype: float64


In [39]:
market_counts = df["target_markets"].apply(lambda x: len(x) if isinstance(x,list) else 0)

In [40]:
print(f"Items per company in target_markets: {market_counts.describe()}")

Items per company in target_markets: count    477.000000
mean       2.750524
std        2.805050
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max       31.000000
Name: target_markets, dtype: float64


In [41]:
print(df["is_public"].value_counts())

is_public
False    436
True      41
Name: count, dtype: int64


In [42]:
df["business_model"]

0      [Wholesale, Manufacturing, Business-to-Busines...
1      [Retail, Business-to-Business, Business-to-Con...
2      [Wholesale, Manufacturing, Business-to-Busines...
3      [Service Provider, Business-to-Business, Enter...
4      [Business-to-Business, Enterprise, Manufacturi...
                             ...                        
472    [Service Provider, Business-to-Business, Enter...
473                [Business-to-Business, Manufacturing]
474     [Business-to-Business, Manufacturing, Licensing]
475    [Service Provider, Business-to-Business, Enter...
476     [Business-to-Business, Manufacturing, Wholesale]
Name: business_model, Length: 477, dtype: object

In [43]:
all_models = []
for bm in df["business_model"]:
    if isinstance(bm,list):
        all_models.extend(bm)

In [44]:
for model, count in Counter(all_models).most_common():
    print(f"{model}:{count}")

Business-to-Business:441
Manufacturing:253
Service Provider:237
Enterprise:159
Business-to-Consumer:124
Software-as-a-Service:90
Wholesale:89
Retail:74
E-commerce:38
Subscription-Based:16
Licensing:8
Government/Public Sector:3
Logistics/Transportation:3
Real Estate Development:3
Non-Profit/Non-Governmental Organization:3
Asset Management:2
Franchising:1
Entertainment:1


In [47]:
key_fields = ["description","primary_naics","address","employee_count",
              "revenue","business_model","core_offerings","target_markets",
              "year_founded","is_public"]

In [48]:
df["fields_present"] = df[key_fields].notna().sum(axis=1)

In [49]:
print(f"Fields present per company (out of 10): {df["fields_present"].value_counts().sort_index()}")

Fields present per company (out of 10): fields_present
7      37
8     104
9      93
10    243
Name: count, dtype: int64
